<a href="https://colab.research.google.com/github/JakeOh/202605_BD57/blob/main/lab_ml/ml03_train_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier  # KNN 분류기 머신러닝 모델
from sklearn.model_selection import train_test_split  # 훈련 셋/테스트 셋 분리 함수
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score  # 평가지표 함수들
from sklearn.metrics import classification_report, confusion_matrix  # 평가지표 내용 출력
from sklearn.preprocessing import StandardScaler  # 표준화 특성 스케일 변환기

# 데이터 준비

In [2]:
file_path = 'https://bit.ly/fish_csv_data'
# https://raw.githubusercontent.com/rickiepark/hg-mldl/master/fish.csv

In [3]:
fish = pd.read_csv(file_path)

In [4]:
fish.head()

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340


In [5]:
fish.tail()

,Species,Weight,Length,Diagonal,Height,Width
154,Smelt,12.2,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,14.3,15.2,2.8728,2.0672
158,Smelt,19.9,15.0,16.2,2.9322,1.8792


In [6]:
pd.set_option('display.max_rows', 10)

In [7]:
# Bream(도미) vs Smelt(빙어) 분류 문제
df = fish[fish.Species.isin(['Bream', 'Smelt'])]
df

,Species,Weight,Length,Diagonal,Height,Width
0,Bream,242.0,25.4,30.0,11.5200,4.0200
1,Bream,290.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,26.5,31.1,12.3778,4.6961
3,Bream,363.0,29.0,33.5,12.7300,4.4555
4,Bream,430.0,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...
154,Smelt,12.2,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,14.3,15.2,2.8728,2.0672


In [10]:
# 특성(features) 배열 - 2차원
x = df[['Weight', 'Length']].values

In [11]:
x[:5]

array([[242. ,  25.4],
       [290. ,  26.3],
       [340. ,  26.5],
       [363. ,  29. ],
       [430. ,  29. ]])

In [12]:
x[-5:]

array([[12.2, 12.2],
       [13.4, 12.4],
       [12.2, 13. ],
       [19.7, 14.3],
       [19.9, 15. ]])

In [13]:
x.shape  #> (n_samples, n_features)

(49, 2)

In [14]:
# 타겟 배열 - 1차원 배열
y = df.Species.values

In [15]:
y

array(['Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
       'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
       'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
       'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
       'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream', 'Bream',
       'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt',
       'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt', 'Smelt'],
      dtype=object)

# 훈련 셋 vs 테스트 셋 분리

*  머신러닝(Machine Learning): 컴퓨터가 데이터를 학습해서 예측.
*  훈련(training), 학습(learning): 머신러닝 알고리즘(모델)에게 특성 데이터(와 레이블)을 제공.
*  평가(evaluation): 학습된 데이터와 학습되지 않은 데이터를 얼마나 잘 예측하는 지를 점수화.
*  훈련 셋: 머신러닝 모델에게 제공하는 특성 데이터(와 레이블)
*  테스트 셋: 평가를 위해서만 남겨두는 특성 데이터(와 레이블)
*  훈련 셋과 테스트 셋으로 분리하는 방법
    *   순차 추출(sequential sampling)
    *   임의 추출(random sampling)
    *   층화 추출(stratified sampling)

## 순차 추출(sequential sampling)

In [17]:
num_trains = 35  # 훈련 셋 샘플 개수(전체 샘플의 약 70%)

In [19]:
# 특성 배열을 훈련 셋, 테스트 셋으로 분리
x_train = x[:num_trains]  # 훈련 셋
x_test = x[num_trains:]  # 테스트 셋

In [20]:
# 레이블(타겟 배열) 훈련 레이블, 테스트 레이블로 분리
y_train = y[:num_trains]  # 훈련 레이블
y_test = y[num_trains:]  # 테스트 레이블

In [21]:
x_train.shape  #> (35, 2) = (n_samples, n_features)

(35, 2)

In [22]:
x_test.shape  #> (14, 2)

(14, 2)

In [23]:
y_train.shape  #> (35,) = (n_samples,)

(35,)

In [24]:
y_test.shape  #> (14,)

(14,)

In [25]:
# KNN 분류 모델 생성
knn = KNeighborsClassifier()

In [26]:
# KNN 모델을 훈련 셋(과 훈련 레이블)로 훈련
knn.fit(X=x_train, y=y_train)

KNeighborsClassifier()

In [27]:
# 학습된 데이터(훈련 셋)로 평가
train_acc = knn.score(X=x_train, y=y_train)
train_acc

1.0

In [28]:
# 학습되지 않은 데이터(테스트 셋)에서 평가
test_acc = knn.score(X=x_test, y=y_test)
test_acc

0.0